In [1]:
import pandas as pd
from pathlib import Path

# ============================================================
# Config
# ============================================================
BASE_DIR = Path("./")

INPUT_QUANT = BASE_DIR / "event_quantification_window0_summary.csv"
INPUT_PERSIST = BASE_DIR / "event_persistence_outputs" / "event_persistence_window0.csv"

OUT_DIR = BASE_DIR / "event_representation_outputs"
OUT_DIR.mkdir(exist_ok=True)

OUT_FILE = OUT_DIR / "event_representation_matrix_window0.csv"

# ============================================================
# 1. Load
# ============================================================
df_q = pd.read_csv(INPUT_QUANT)
df_p = pd.read_csv(INPUT_PERSIST)

df_q["Date"] = pd.to_datetime(df_q["Date"])
df_p["Date"] = pd.to_datetime(df_p["Date"])

# ============================================================
# 2. Merge
# ============================================================
df = pd.merge(df_q, df_p, on="Date", how="inner")

# ============================================================
# 3. Feature 선택
# ============================================================
feature_cols = [
    "Magnitude",
    "Top1_DeltaS",
    "Top2_DeltaS",
    "Top3_DeltaS",
    "Peak",
    "PeakHorizon",
    "HalfLife",
    "DurationAboveHalf",
    "AUC"
]

df_feat = df[["Date"] + feature_cols].copy()

# ============================================================
# 4. 결측 처리
# ============================================================
df_feat = df_feat.dropna().reset_index(drop=True)

# ============================================================
# 5. Scaling (선택)
# ============================================================
# clustering 대비 z-score
scale_cols = feature_cols

df_scaled = df_feat.copy()

for col in scale_cols:
    mean = df_scaled[col].mean()
    std = df_scaled[col].std()
    if std == 0:
        df_scaled[col] = 0.0
    else:
        df_scaled[col] = (df_scaled[col] - mean) / std

# ============================================================
# 6. Save
# ============================================================
df_feat.to_csv(OUT_DIR / "event_representation_matrix_window0_raw.csv", index=False)
df_scaled.to_csv(OUT_DIR / "event_representation_matrix_window0_scaled.csv", index=False)

print("[INFO] Saved raw matrix:", OUT_DIR / "event_representation_matrix_window0_raw.csv")
print("[INFO] Saved scaled matrix:", OUT_DIR / "event_representation_matrix_window0_scaled.csv")
print("[INFO] Done")

[INFO] Saved raw matrix: event_representation_outputs\event_representation_matrix_window0_raw.csv
[INFO] Saved scaled matrix: event_representation_outputs\event_representation_matrix_window0_scaled.csv
[INFO] Done
